# 11 - Finalisasi Data Training

Gabungin `dataset_final.csv` hasil tahap 10 (sudah di-split train/test + fold, sudah diaugmentasi
negatif) dengan sintesis paraphrase & dummy data manual, jadi 2 set file training SIAP PAKAI untuk
fine-tuning IndoBERTweet:

- `output/11_finalisasi_data_training/3class/{train.csv, test.csv}` (skema 3 kelas)
- `output/11_finalisasi_data_training/binary/{train.csv, test.csv}` (skema binary)

**Aturan anti-leakage:** TEST set SELALU cuma baris asli (`split=='test'`), ga pernah ditambah
sintesis/dummy, dan SAMA PERSIS dipakai di 2 skema biar hasil keduanya bisa dibandingin fair.
Sintesis paraphrase & dummy cuma ditempel ke TRAIN.

Sumber file sintesis/dummy TIDAK ADA script generator-nya di repo (hasil kerja manual/LLM-assisted) --
jadi disalin apa adanya ke folder output tahap ini.

In [ ]:
from pathlib import Path

import pandas as pd

# -------------------------------------------------------------------------
# KONFIGURASI PATH
# -------------------------------------------------------------------------
FINAL_OUTPUT_ROOT = Path.cwd() / "output"
OUTPUT_DIR = FINAL_OUTPUT_ROOT / "11_finalisasi_data_training"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = FINAL_OUTPUT_ROOT / "10_augmentasi_data_negatif" / "dataset_final.csv"

SINTESIS_NEGATIF_PATH = OUTPUT_DIR / "sintesis_negatif_train.csv"
DUMMY_NEGATIF_PATH = OUTPUT_DIR / "dummy_negatif_baru.csv"
SINTESIS_POSITIF_PATH = OUTPUT_DIR / "sintesis_positif_train.csv"
DUMMY_POSITIF_PATH = OUTPUT_DIR / "dummy_positif_baru.csv"

OUT_3CLASS_DIR = OUTPUT_DIR / "3class"
OUT_BINARY_DIR = OUTPUT_DIR / "binary"
print("Config siap.")

In [ ]:
def _wajib_ada(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"{path} tidak ditemukan.")
    return path


def _load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(_wajib_ada(path), dtype=str, keep_default_na=False, encoding="utf-8")


def _gabung(daftar_df: list) -> pd.DataFrame:
    """Concat dengan union kolom -- kolom yang ga ada di suatu df otomatis diisi string kosong."""
    hasil = pd.concat(daftar_df, ignore_index=True, sort=False)
    return hasil.fillna("")

## Fungsi: gabung dataset asli (tahap 10) + sintesis/dummy -> train/test final

In [ ]:
def gabung_final_training() -> None:
    out_files = [OUT_3CLASS_DIR / "train.csv", OUT_3CLASS_DIR / "test.csv",
                 OUT_BINARY_DIR / "train.csv", OUT_BINARY_DIR / "test.csv"]
    if all(p.exists() for p in out_files):
        print("[Gabung] Semua file train/test sudah ada -> skip")
        return

    df = _load_csv(DATASET_PATH)
    required = {"split", "label_final", "label_name", "comment_text_clean"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Kolom berikut tidak ada di dataset_final.csv: {missing}")
    if not (df["split"] == "train").any() or not (df["split"] == "test").any():
        raise ValueError("Kolom 'split' belum lengkap keisi 'train'/'test'. Jalankan tahap 10 (split_dataset) dulu.")

    df_asli = df.copy()
    df_asli["text"] = df_asli["comment_text_clean"]
    df_asli["sumber"] = "asli"

    train_asli = df_asli[df_asli["split"] == "train"].reset_index(drop=True)
    test_asli = df_asli[df_asli["split"] == "test"].reset_index(drop=True)

    sintesis_negatif = _load_csv(SINTESIS_NEGATIF_PATH)
    sintesis_negatif["text"] = sintesis_negatif["paraphrase_text"]
    sintesis_negatif["sumber"] = "sintesis_paraphrase"

    dummy_negatif = _load_csv(DUMMY_NEGATIF_PATH)
    dummy_negatif["sumber"] = "dummy_baru"

    sintesis_positif = _load_csv(SINTESIS_POSITIF_PATH)
    sintesis_positif["text"] = sintesis_positif["paraphrase_text"]
    sintesis_positif["sumber"] = "sintesis_paraphrase"

    dummy_positif = _load_csv(DUMMY_POSITIF_PATH)
    dummy_positif["sumber"] = "dummy_baru"

    for nama, d, label_expected in [
        ("sintesis_negatif", sintesis_negatif, "negatif"),
        ("dummy_negatif", dummy_negatif, "negatif"),
        ("sintesis_positif", sintesis_positif, "positif"),
        ("dummy_positif", dummy_positif, "positif"),
    ]:
        if not (d["label_final"] == label_expected).all():
            raise ValueError(f"File {nama} ada baris dengan label_final != '{label_expected}', cek lagi datanya.")

    train_3class = _gabung([train_asli, sintesis_negatif, dummy_negatif, sintesis_positif, dummy_positif])
    test_3class = test_asli.copy()

    train_binary = _gabung([train_asli, sintesis_negatif, dummy_negatif])
    test_binary = test_asli.copy()

    for d in (OUT_3CLASS_DIR, OUT_BINARY_DIR):
        d.mkdir(parents=True, exist_ok=True)

    train_3class.to_csv(OUT_3CLASS_DIR / "train.csv", index=False, encoding="utf-8")
    test_3class.to_csv(OUT_3CLASS_DIR / "test.csv", index=False, encoding="utf-8")
    train_binary.to_csv(OUT_BINARY_DIR / "train.csv", index=False, encoding="utf-8")
    test_binary.to_csv(OUT_BINARY_DIR / "test.csv", index=False, encoding="utf-8")

    print(f"[Gabung] 3class -> train={len(train_3class)}, test={len(test_3class)}")
    print(f"[Gabung] binary -> train={len(train_binary)}, test={len(test_binary)}")

## Jalankan: gabung data

In [ ]:
gabung_final_training()

## Fungsi: stats/laporan per skema (tidak nulis file baru, cuma print)

In [ ]:
FILES_PER_SKEMA = {
    "3class": {"train": OUT_3CLASS_DIR / "train.csv", "test": OUT_3CLASS_DIR / "test.csv",
                "label_col": "label_final"},
    "binary": {"train": OUT_BINARY_DIR / "train.csv", "test": OUT_BINARY_DIR / "test.csv",
               "label_col": "label_name"},
}


def show_stats_scheme(skema: str) -> None:
    paths = FILES_PER_SKEMA[skema]
    if not paths["train"].exists() or not paths["test"].exists():
        print(f"[Stats] {skema}: belum ada, jalankan gabung_final_training() dulu.")
        return

    df_train = _load_csv(paths["train"])
    df_test = _load_csv(paths["test"])
    label_col = paths["label_col"]

    required = {"text", "sumber", label_col}
    for nama_df, d in [("train", df_train), ("test", df_test)]:
        missing = required - set(d.columns)
        if missing:
            raise ValueError(f"File {skema}/{nama_df}.csv kurang kolom: {missing}")

    for nama, d in [(f"{skema.upper()} - TRAIN", df_train), (f"{skema.upper()} - TEST", df_test)]:
        print(f"\n{'=' * 55}\n{nama}  (total {len(d)} baris)\n{'=' * 55}")

        print(f"-- Proporsi {label_col} --")
        for label, jumlah in d[label_col].value_counts().items():
            pct = 100 * jumlah / len(d) if len(d) else 0
            print(f"  {label:14s}: {jumlah:5d}  ({pct:5.1f}%)")

        print("-- Proporsi sumber --")
        for sumber, jumlah in d["sumber"].value_counts().items():
            pct = 100 * jumlah / len(d) if len(d) else 0
            print(f"  {sumber:20s}: {jumlah:5d}  ({pct:5.1f}%)")

        print("-- Panjang teks (jumlah kata) per label --")
        d2 = d.copy()
        d2["_jumlah_kata"] = d2["text"].str.split().apply(len)
        stats = d2.groupby(label_col)["_jumlah_kata"].agg(["mean", "min", "max"])
        for label, row in stats.iterrows():
            print(f"  {label:14s}: rata-rata {row['mean']:.1f} kata (min {int(row['min'])}, max {int(row['max'])})")

        kosong = d["text"].str.strip().eq("")
        if kosong.any():
            print(f"  PERHATIAN: {nama} punya {kosong.sum()} baris dengan teks kosong!")

    print(f"\n-- Perbandingan SEBELUM vs SESUDAH augmentasi (TRAIN {skema}, label {label_col}) --")
    train_asli_only = df_train[df_train["sumber"] == "asli"]
    labels = sorted(set(train_asli_only[label_col]) | set(df_train[label_col]))
    n_before, n_after = len(train_asli_only), len(df_train)
    print(f"  {'label':14s} {'SEBELUM (asli)':>20s} {'SESUDAH (+sintesis)':>22s} {'selisih':>10s}")
    for label in labels:
        c_before = int((train_asli_only[label_col] == label).sum())
        c_after = int((df_train[label_col] == label).sum())
        pct_before = 100 * c_before / n_before if n_before else 0
        pct_after = 100 * c_after / n_after if n_after else 0
        print(f"  {label:14s} {c_before:6d} ({pct_before:5.1f}%)     "
              f"{c_after:6d} ({pct_after:5.1f}%)        +{c_after - c_before}")
    print(f"  {'TOTAL':14s} {n_before:6d} (100.0%)     {n_after:6d} (100.0%)        +{n_after - n_before}")

### Skema 3CLASS

In [ ]:
show_stats_scheme("3class")

### Skema BINARY

In [ ]:
show_stats_scheme("binary")
print("\nSelesai cek komposisi. Kalau semua wajar (ga ada teks kosong, proporsi sesuai ekspektasi), lanjut ke tahap 12 (fine-tuning).")